# LogiFlow-RL — GRPO Training (Phase 2)

**One-click reproducible training notebook for judges.**

This notebook clones the repo, installs dependencies, and runs the production training script (`train_grpo.py`) which:
1. Validates the reward function before training
2. Evaluates baseline policies (round_robin, heuristic) BEFORE training
3. Runs GRPO with TRL + LoRA on Qwen2.5-0.5B-Instruct
4. Evaluates the trained model AFTER training
5. Saves reward curve PNG, before/after CSVs, and improvement JSON

**Total runtime: ~30–45 minutes on T4 GPU.**

**To re-run from scratch: Runtime → Run all.**

In [ ]:
# === Cell 1: Clone repo and install dependencies ===
import os, subprocess, sys
from pathlib import Path

# Update this if your repo URL is different
REPO_URL = "https://github.com/Roshan5105labs/crisis-logistics-env.git"
REPO_ROOT = Path("/content/crisis-logistics-env")

# Clone (or pull if already cloned)
if not REPO_ROOT.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_ROOT)])
else:
    subprocess.call(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"])

# Find the package directory (handles both flat and nested layouts)
candidates = [
    REPO_ROOT / "crisis_logistics_env",
    REPO_ROOT,  # if the repo IS the package
]
PKG_DIR = next((p for p in candidates if (p / "server").exists() and (p / "tasks.py").exists()), None)
if PKG_DIR is None:
    raise FileNotFoundError(f"Could not find package in {REPO_ROOT}")

# Add the parent of the package to sys.path so 'crisis_logistics_env' is importable
sys.path.insert(0, str(PKG_DIR.parent))
os.chdir(PKG_DIR)
print(f"Working from: {PKG_DIR}")
print(f"sys.path[0]: {sys.path[0]}")

In [ ]:
import sys
sys.path.insert(0, "outputs/logiflow-grpo-script")

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16, device_map="auto"
)
adapter_path = "outputs/logiflow-grpo-script"
model = PeftModel.from_pretrained(base, adapter_path)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
if tok.pad_token is None: tok.pad_token = tok.eos_token

# Generate one decision on a hard task
from crisis_logistics_env.server.crisis_logistics_env_environment \
    import CrisisLogisticsEnvironment
env = CrisisLogisticsEnvironment()
obs = env.reset(task_id="hard")
for _ in range(8): obs = env.step(__import__("crisis_logistics_env"
    ).server.crisis_logistics_env_environment.choose_network_action(obs))

prompt = (
    f"Task: {env.task.title}\\n"
    f"Visible nodes: {obs.visible_node_ids}\\n"
    f"Loads: {obs.observed_node_loads}\\n"
    f"Active disruptions: {obs.active_disruptions}\\n"
    f"Incoming: source={obs.pending_source_node}, vol={obs.incoming_load}\\n"
    "Return JSON: {reasoning, source_node, dest_node, shipment_volume}"
)

inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=150, do_sample=False,
                     pad_token_id=tok.eos_token_id)
text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== Trained agent's reasoning ===")
print(text)

In [ ]:
# === Cell 2: Install pinned dependencies ===
# Pin TRL to a known-good version. Newer TRL has GRPOConfig at trl.GRPOConfig.
# If trl 0.12 doesn't have it, fallback to 0.14+
!pip install -q --upgrade pip
!pip install -q \
    "openenv-core[core]>=0.2.3" \
    "trl>=0.14.0,<0.18.0" \
    "transformers>=4.46,<4.50" \
    "accelerate>=0.34" \
    "datasets>=3.0" \
    "peft>=0.13" \
    "bitsandbytes" \
    "matplotlib" \
    "pandas" \
    "pydantic>=2.0"

# Verify the critical imports work
import importlib, sys
for mod in ["trl", "transformers", "peft", "datasets"]:
    try:
        m = importlib.import_module(mod)
        print(f"✓ {mod} {m.__version__}")
    except Exception as e:
        print(f"✗ {mod}: {e}")

# Test that GRPOConfig exists
try:
    from trl import GRPOConfig, GRPOTrainer
    print("✓ GRPOConfig and GRPOTrainer imported successfully")
except ImportError as e:
    print(f"✗ TRL too old. Run !pip install --upgrade 'trl>=0.14' and restart runtime.")
    raise

In [ ]:
# === Cell 3: Verify the environment imports correctly ===
from crisis_logistics_env.server.crisis_logistics_env_environment import CrisisLogisticsEnvironment
from crisis_logistics_env.tasks import list_tasks

env = CrisisLogisticsEnvironment()
obs = env.reset(task_id="easy")
print(f"✓ Environment loaded — task: {obs.task_id}")
print(f"✓ Available tasks: {[t.task_id for t in list_tasks()]}")
print(f"✓ Observation has {len(obs.node_loads)} nodes (expected 12)")
print(f"✓ Visible nodes: {obs.visible_node_ids}")
print(f"✓ First step source: {obs.pending_source_node}, volume: {obs.incoming_load}")

assert len(obs.node_loads) == 12, "Expected 12-node network"
print("\n✓ Environment sanity check PASSED")

In [ ]:
# === Cell 4: Run the production GRPO training script ===
# This runs train_grpo.py which:
#   1. Builds training dataset from environment rollouts
#   2. Sanity-checks the reward function (refuses to start if broken)
#   3. Evaluates round_robin + heuristic baselines BEFORE training
#   4. Runs GRPO training (LoRA on Qwen2.5-0.5B-Instruct)
#   5. Evaluates the trained model AFTER training
#   6. Saves all artifacts

# Reduce max_steps if you're short on time. 200 is the sweet spot for visible learning.
# Use --max-steps 80 for a quick smoke test (~15 min).

!python train_grpo.py \
    --model-name "Qwen/Qwen2.5-0.5B-Instruct" \
    --max-steps 200 \
    --num-generations 4 \
    --samples-per-task 42 \
    --output-dir "outputs/logiflow-grpo-script"

In [ ]:
# === Cell 5: Display training artifacts ===
from pathlib import Path
import json
from IPython.display import Image, display

ARTIFACTS = Path("outputs/logiflow-grpo-script/artifacts")

if not ARTIFACTS.exists():
    print("⚠ Artifacts folder not found. Did training complete?")
    print(f"Looked at: {ARTIFACTS.resolve()}")
else:
    print("=== Generated artifacts ===")
    for f in sorted(ARTIFACTS.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:40s}  {size_kb:>8.1f} KB")
    print()

    # Show reward curve
    curve_path = ARTIFACTS / "reward_curve.png"
    if curve_path.exists():
        print("=== Reward Curve ===")
        display(Image(str(curve_path)))

    # Show improvement summary
    improvement_path = ARTIFACTS / "improvement.json"
    if improvement_path.exists():
        print("\n=== Improvement Summary ===")
        with open(improvement_path) as f:
            data = json.load(f)
        print(json.dumps(data, indent=2))

    # Show evaluation summary
    eval_path = ARTIFACTS / "evaluation_summary.json"
    if eval_path.exists():
        print("\n=== Evaluation Summary (before vs after) ===")
        with open(eval_path) as f:
            data = json.load(f)
        print(json.dumps(data, indent=2))

In [ ]:
# === Cell 6: Before vs After comparison table ===
import pandas as pd
from pathlib import Path

ARTIFACTS = Path("outputs/logiflow-grpo-script/artifacts")

before_csv = ARTIFACTS / "evaluation_before.csv"
after_csv = ARTIFACTS / "evaluation_after.csv"

if before_csv.exists() and after_csv.exists():
    before = pd.read_csv(before_csv)
    after = pd.read_csv(after_csv)

    print("=== BEFORE TRAINING ===")
    print(before.to_string(index=False))
    print("\n=== AFTER TRAINING ===")
    print(after.to_string(index=False))

    # Build comparison if model policy exists in both
    if 'policy' in before.columns and 'policy' in after.columns:
        merged = before.merge(after, on=['policy', 'task_id'], suffixes=('_before', '_after'))
        merged['score_delta'] = merged['score_after'] - merged['score_before']
        print("\n=== DELTA ===")
        print(merged[['policy', 'task_id', 'score_before', 'score_after', 'score_delta']].to_string(index=False))
else:
    print("Evaluation CSVs not found — check training logs.")

## What just happened

1. **Cloned and installed** the LogiFlow-RL package and dependencies
2. **Verified** the 12-node environment loads correctly
3. **Ran `train_grpo.py`** which is the production training pipeline:
   - Built a dataset of prompts from environment rollouts
   - Validated the reward function (sanity check)
   - Evaluated `round_robin` and `heuristic` baselines
   - Trained Qwen2.5-0.5B with GRPO + LoRA for 200 steps
   - Evaluated the trained model on all 3 tasks
   - Saved 6 artifact files for the README and blog
4. **Displayed** the reward curve, improvement JSON, and before/after comparison

## Files generated

| File | Purpose |
|------|---------|
| `reward_curve.png` | Training reward over time — embed in README |
| `evaluation_before.csv` | Baseline scores before training |
| `evaluation_after.csv` | Trained model scores |
| `evaluation_summary.csv` | Both phases combined |
| `evaluation_summary.json` | Same data as JSON |
| `improvement.json` | Score deltas |

## To submit

Download all files from `outputs/logiflow-grpo-script/artifacts/` and commit them to the repo. Embed `reward_curve.png` in your README.